In [2]:
!pip install transformers==4.41.0 -q
print('Instalado.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.0 MB/s eta 0:00:0000:01
Instalado.


In [3]:
import torch
import transformers
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0))


torch: 2.10.0+cu128
transformers: 4.41.0
GPU: Tesla T4


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [5]:
train = pd.read_csv('/kaggle/input/datasets/beleniniestatejera/datostrabajo/train_clean.csv')
test = pd.read_csv('/kaggle/input/datasets/beleniniestatejera/datostrabajo/test_nolabel.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print()
print('Distribución label:')
print(train['label'].value_counts())

Train shape: (8947, 10)
Test shape: (3836, 7)

Distribución label:
label
1    5794
0    3153
Name: count, dtype: int64


In [6]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def preparar_texto(df):
    textos = []
    for _, row in df.iterrows():
        texto = f"Speaker: {row['speaker']}. Party: {row['party_affiliation']}. Statement: {row['statement']}"
        textos.append(texto)
    return textos

train_textos = preparar_texto(train)
test_textos = preparar_texto(test)

print(f'Ejemplo:')
print(train_textos[0])
print()
print(f'Total train: {len(train_textos)}')
print(f'Total test: {len(test_textos)}')


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Ejemplo:
Speaker: donald-trump. Party: republican. Statement: China is in the South China Sea and a military fortress the likes of which perhaps the world has not seen.

Total train: 8947
Total test: 3836


In [7]:
class FakeNewsDataset(Dataset):
    def __init__(self, textos, labels=None, tokenizer=None, max_length=128):
        self.textos = textos
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.textos[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Split train/validation
train_textos_tr, train_textos_val, y_tr, y_val = train_test_split(
    train_textos, train['label'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=train['label']
)

# Crear datasets
train_dataset = FakeNewsDataset(train_textos_tr, y_tr, tokenizer)
val_dataset = FakeNewsDataset(train_textos_val, y_val, tokenizer)
test_dataset = FakeNewsDataset(test_textos, tokenizer=tokenizer)

# Crear dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Train: {len(train_dataset)} ejemplos, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} ejemplos, {len(val_loader)} batches')
print(f'Test:  {len(test_dataset)} ejemplos, {len(test_loader)} batches')

Train: 7604 ejemplos, 476 batches
Val:   1343 ejemplos, 42 batches
Test:  3836 ejemplos, 120 batches


In [8]:
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2
)
model = model.to(device)

# Optimizador y scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

epochs = 4
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f'Modelo cargado en: {device}')
print(f'Parámetros totales: {sum(p.numel() for p in model.parameters()):,}')
print(f'Total steps: {total_steps}')
print(f'Warmup steps: {int(0.1 * total_steps)}')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelo cargado en: cuda
Parámetros totales: 124,647,170
Total steps: 1904
Warmup steps: 190


In [11]:
def evaluar(model, loader):
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            
            preds.extend(predictions.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    
    f1 = f1_score(labels_all, preds, average='macro')
    return f1, preds, labels_all

# Entrenamiento — continuar desde epoch 2 ya que epoch 1 completó el forward
best_f1 = 0
best_preds_val = None

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    
    avg_loss = total_loss / len(train_loader)
    val_f1, _, _ = evaluar(model, val_loader)
    
    print(f'\nEpoch {epoch+1} completado | Avg Loss: {avg_loss:.4f} | Val F1-Macro: {val_f1:.4f}')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'Mejor modelo guardado! F1: {best_f1:.4f}')
    print()

Epoch 1/4 | Batch 0/476 | Loss: 0.7072
Epoch 1/4 | Batch 100/476 | Loss: 0.6477
Epoch 1/4 | Batch 200/476 | Loss: 0.6685
Epoch 1/4 | Batch 300/476 | Loss: 0.5921
Epoch 1/4 | Batch 400/476 | Loss: 0.5479

Epoch 1 completado | Avg Loss: 0.6465 | Val F1-Macro: 0.5935
Mejor modelo guardado! F1: 0.5935

Epoch 2/4 | Batch 0/476 | Loss: 0.6449
Epoch 2/4 | Batch 100/476 | Loss: 0.6044
Epoch 2/4 | Batch 200/476 | Loss: 0.6230
Epoch 2/4 | Batch 300/476 | Loss: 0.5947
Epoch 2/4 | Batch 400/476 | Loss: 0.5892

Epoch 2 completado | Avg Loss: 0.5959 | Val F1-Macro: 0.6214
Mejor modelo guardado! F1: 0.6214

Epoch 3/4 | Batch 0/476 | Loss: 0.4791
Epoch 3/4 | Batch 100/476 | Loss: 0.5894
Epoch 3/4 | Batch 200/476 | Loss: 0.7482
Epoch 3/4 | Batch 300/476 | Loss: 0.6049
Epoch 3/4 | Batch 400/476 | Loss: 0.5527

Epoch 3 completado | Avg Loss: 0.5242 | Val F1-Macro: 0.6227
Mejor modelo guardado! F1: 0.6227

Epoch 4/4 | Batch 0/476 | Loss: 0.5491
Epoch 4/4 | Batch 100/476 | Loss: 0.5715
Epoch 4/4 | Batch 20

In [12]:
# Cargar el mejor modelo
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# Predecir en test
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(predictions.cpu().tolist())

# Crear submission
submission = pd.DataFrame({
    'id': test['id'],
    'label': all_preds
})

submission.to_csv('submission_roberta.csv', index=False)
print('Submission guardada como submission_roberta.csv')
print()
print('Distribución de predicciones:')
print(submission['label'].value_counts())
print()
print('Resumen final:')
print(f'  Modelo:       RoBERTa-base')
print(f'  Epochs:       4')
print(f'  Best epoch:   3')
print(f'  F1-Macro Val: 0.6377')

Submission guardada como submission_roberta.csv

Distribución de predicciones:
label
1    2443
0    1393
Name: count, dtype: int64

Resumen final:
  Modelo:       RoBERTa-base
  Epochs:       4
  Best epoch:   3
  F1-Macro Val: 0.6377


In [13]:
import torch.nn.functional as F

model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# Probabilidades en test
proba_roberta_test = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        proba_roberta_test.extend(probs[:, 1].cpu().tolist())

# Probabilidades en validación para calibrar el ensemble
proba_roberta_val = []
labels_val_all = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        proba_roberta_val.extend(probs[:, 1].cpu().tolist())
        labels_val_all.extend(labels.cpu().tolist())

proba_roberta_test = np.array(proba_roberta_test)
proba_roberta_val = np.array(proba_roberta_val)
labels_val_all = np.array(labels_val_all)

print(f'Probabilidades test shape: {proba_roberta_test.shape}')
print(f'Probabilidades val shape: {proba_roberta_val.shape}')

Probabilidades test shape: (3836,)
Probabilidades val shape: (1343,)


In [18]:
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

# Reconstruir NB mejorado
MIN_COUNT = 5
vc = train['speaker'].value_counts()
speakers_frecuentes = vc[vc >= MIN_COUNT].index
test_speaker_grouped = test['speaker'].where(test['speaker'].isin(speakers_frecuentes), other='other')

vc_party = train['party_affiliation'].value_counts()
minoritarias = vc_party[vc_party < 50].index
test_party_grouped = test['party_affiliation'].where(~test['party_affiliation'].isin(minoritarias), other='other')
test_subject_primary = test['subject'].str.split(',').str[0].str.strip()

train_subject_primary = train['subject_primary']
train_speaker_grouped = train['speaker_grouped']
train_party_grouped = train['party_grouped']

cat_cols = ['speaker_grouped', 'party_grouped', 'subject_primary']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = ohe.fit_transform(train[cat_cols])

test_cat_df = pd.DataFrame({
    'speaker_grouped': test_speaker_grouped,
    'party_grouped': test_party_grouped,
    'subject_primary': test_subject_primary
})
X_test_cat = ohe.transform(test_cat_df)

vect_nb = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=7)
X_train_nb = hstack([vect_nb.fit_transform(train['statement']), X_train_cat])
X_test_nb = hstack([vect_nb.transform(test['statement']), X_test_cat])

nb_model = ComplementNB(alpha=3.5, norm=False)
nb_model.fit(X_train_nb, train['label'])
proba_nb_test = nb_model.predict_proba(X_test_nb)[:, 1]

print('NB probabilidades test shape:', proba_nb_test.shape)
print()

# Probar pesos
print('Ensemble RoBERTa + NB:')
for w_rob in [0.7, 0.8, 0.9, 0.95]:
    w_nb = 1 - w_rob
    proba_ens = w_rob * proba_roberta_test + w_nb * proba_nb_test
    for threshold in [0.45, 0.50, 0.55]:
        y_pred = (proba_ens >= threshold).astype(int)
        f1 = f1_score(labels_val_all, 
                      (w_rob * proba_roberta_val + w_nb * nb_model.predict_proba(X_train_nb[-len(labels_val_all):])[: ,1] >= threshold).astype(int),
                      average='macro')
        print(f'RoBERTa={w_rob:.2f} NB={w_nb:.2f} threshold={threshold}: F1-Val={f1:.4f}')
    print()

NB probabilidades test shape: (3836,)

Ensemble RoBERTa + NB:
RoBERTa=0.70 NB=0.30 threshold=0.45: F1-Val=0.6236
RoBERTa=0.70 NB=0.30 threshold=0.5: F1-Val=0.6334
RoBERTa=0.70 NB=0.30 threshold=0.55: F1-Val=0.6351

RoBERTa=0.80 NB=0.20 threshold=0.45: F1-Val=0.6285
RoBERTa=0.80 NB=0.20 threshold=0.5: F1-Val=0.6398
RoBERTa=0.80 NB=0.20 threshold=0.55: F1-Val=0.6398

RoBERTa=0.90 NB=0.10 threshold=0.45: F1-Val=0.6326
RoBERTa=0.90 NB=0.10 threshold=0.5: F1-Val=0.6401
RoBERTa=0.90 NB=0.10 threshold=0.55: F1-Val=0.6463

RoBERTa=0.95 NB=0.05 threshold=0.45: F1-Val=0.6369
RoBERTa=0.95 NB=0.05 threshold=0.5: F1-Val=0.6426
RoBERTa=0.95 NB=0.05 threshold=0.55: F1-Val=0.6433



In [19]:
# Ensemble óptimo: RoBERTa=0.90, NB=0.10, threshold=0.55
proba_ensemble_final = 0.90 * proba_roberta_test + 0.10 * proba_nb_test
y_pred_ensemble = (proba_ensemble_final >= 0.55).astype(int)

submission = pd.DataFrame({
    'id': test['id'],
    'label': y_pred_ensemble
})

submission.to_csv('submission_roberta_ensemble.csv', index=False)
print('Submission guardada como submission_roberta_ensemble.csv')
print()
print('Distribución de predicciones:')
print(submission['label'].value_counts())
print()
print('Resumen final:')
print(f'  Modelo:        Ensemble RoBERTa-base + Naive Bayes')
print(f'  Pesos:         RoBERTa=0.90, NB=0.10')
print(f'  Threshold:     0.55')
print(f'  F1-Macro Val:  0.6463')

Submission guardada como submission_roberta_ensemble.csv

Distribución de predicciones:
label
1    2278
0    1558
Name: count, dtype: int64

Resumen final:
  Modelo:        Ensemble RoBERTa-base + Naive Bayes
  Pesos:         RoBERTa=0.90, NB=0.10
  Threshold:     0.55
  F1-Macro Val:  0.6463


In [20]:
# Cargar mejor modelo y entrenar 2 epochs más
model.load_state_dict(torch.load('best_model.pt'))

optimizer2 = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)  # lr más bajo
extra_epochs = 2
total_steps2 = len(train_loader) * extra_epochs
scheduler2 = get_linear_schedule_with_warmup(
    optimizer2,
    num_warmup_steps=0,
    num_training_steps=total_steps2
)

best_f1_v2 = 0.6377  # mejor f1 anterior

for epoch in range(extra_epochs):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer2.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer2.step()
        scheduler2.step()
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch+5}/{4+extra_epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    
    avg_loss = total_loss / len(train_loader)
    val_f1, _, _ = evaluar(model, val_loader)
    print(f'\nEpoch {epoch+5} completado | Avg Loss: {avg_loss:.4f} | Val F1-Macro: {val_f1:.4f}')
    
    if val_f1 > best_f1_v2:
        best_f1_v2 = val_f1
        torch.save(model.state_dict(), 'best_model_v2.pt')
        print(f'Mejor modelo guardado! F1: {best_f1_v2:.4f}')
    print()

Epoch 5/6 | Batch 0/476 | Loss: 0.3745
Epoch 5/6 | Batch 100/476 | Loss: 0.3778
Epoch 5/6 | Batch 200/476 | Loss: 0.2246
Epoch 5/6 | Batch 300/476 | Loss: 0.2826
Epoch 5/6 | Batch 400/476 | Loss: 0.6431

Epoch 5 completado | Avg Loss: 0.4048 | Val F1-Macro: 0.6375

Epoch 6/6 | Batch 0/476 | Loss: 0.1781
Epoch 6/6 | Batch 100/476 | Loss: 0.3519
Epoch 6/6 | Batch 200/476 | Loss: 0.1699
Epoch 6/6 | Batch 300/476 | Loss: 0.1412
Epoch 6/6 | Batch 400/476 | Loss: 0.0401

Epoch 6 completado | Avg Loss: 0.2925 | Val F1-Macro: 0.6306



In [21]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Tokenizador DistilBERT
tokenizer_distilbert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Recrear datasets con nuevo tokenizador
class FakeNewsDataset2(Dataset):
    def __init__(self, textos, labels=None, tokenizer=None, max_length=128):
        self.textos = textos
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.textos[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset_db = FakeNewsDataset2(train_textos_tr, y_tr, tokenizer_distilbert)
val_dataset_db = FakeNewsDataset2(train_textos_val, y_val, tokenizer_distilbert)
test_dataset_db = FakeNewsDataset2(test_textos, tokenizer=tokenizer_distilbert)

train_loader_db = DataLoader(train_dataset_db, batch_size=16, shuffle=True)
val_loader_db = DataLoader(val_dataset_db, batch_size=32, shuffle=False)
test_loader_db = DataLoader(test_dataset_db, batch_size=32, shuffle=False)

print(f'Train: {len(train_dataset_db)} ejemplos')
print(f'Val:   {len(val_dataset_db)} ejemplos')
print(f'Test:  {len(test_dataset_db)} ejemplos')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Train: 7604 ejemplos
Val:   1343 ejemplos
Test:  3836 ejemplos


In [22]:
model_db = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model_db = model_db.to(device)

optimizer_db = AdamW(model_db.parameters(), lr=2e-5, weight_decay=0.01)

epochs_db = 4
total_steps_db = len(train_loader_db) * epochs_db
scheduler_db = get_linear_schedule_with_warmup(
    optimizer_db,
    num_warmup_steps=int(0.1 * total_steps_db),
    num_training_steps=total_steps_db
)

print(f'Parámetros totales: {sum(p.numel() for p in model_db.parameters()):,}')

best_f1_db = 0

for epoch in range(epochs_db):
    model_db.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(train_loader_db):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer_db.zero_grad()
        outputs = model_db(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model_db.parameters(), 1.0)
        optimizer_db.step()
        scheduler_db.step()
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch+1}/{epochs_db} | Batch {batch_idx}/{len(train_loader_db)} | Loss: {loss.item():.4f}')
    
    avg_loss = total_loss / len(train_loader_db)
    val_f1, _, _ = evaluar(model_db, val_loader_db)
    
    print(f'\nEpoch {epoch+1} completado | Avg Loss: {avg_loss:.4f} | Val F1-Macro: {val_f1:.4f}')
    
    if val_f1 > best_f1_db:
        best_f1_db = val_f1
        torch.save(model_db.state_dict(), 'best_model_db.pt')
        print(f'Mejor modelo guardado! F1: {best_f1_db:.4f}')
    print()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parámetros totales: 66,955,010
Epoch 1/4 | Batch 0/476 | Loss: 0.7065
Epoch 1/4 | Batch 100/476 | Loss: 0.6209
Epoch 1/4 | Batch 200/476 | Loss: 0.8235
Epoch 1/4 | Batch 300/476 | Loss: 0.5341
Epoch 1/4 | Batch 400/476 | Loss: 0.7141

Epoch 1 completado | Avg Loss: 0.6321 | Val F1-Macro: 0.5618
Mejor modelo guardado! F1: 0.5618

Epoch 2/4 | Batch 0/476 | Loss: 0.5493
Epoch 2/4 | Batch 100/476 | Loss: 0.6392
Epoch 2/4 | Batch 200/476 | Loss: 0.5317
Epoch 2/4 | Batch 300/476 | Loss: 0.6276
Epoch 2/4 | Batch 400/476 | Loss: 0.3970

Epoch 2 completado | Avg Loss: 0.5801 | Val F1-Macro: 0.6301
Mejor modelo guardado! F1: 0.6301

Epoch 3/4 | Batch 0/476 | Loss: 0.5162
Epoch 3/4 | Batch 100/476 | Loss: 0.4761
Epoch 3/4 | Batch 200/476 | Loss: 0.3292
Epoch 3/4 | Batch 300/476 | Loss: 0.7122
Epoch 3/4 | Batch 400/476 | Loss: 0.4951

Epoch 3 completado | Avg Loss: 0.4995 | Val F1-Macro: 0.6165

Epoch 4/4 | Batch 0/476 | Loss: 0.3647
Epoch 4/4 | Batch 100/476 | Loss: 0.3558
Epoch 4/4 | Batch 200/4

In [23]:
# Cargar mejor modelo DistilBERT
model_db.load_state_dict(torch.load('best_model_db.pt'))
model_db.eval()

# Predecir en test
preds_db = []
proba_db_test = []
with torch.no_grad():
    for batch in test_loader_db:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model_db(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(outputs.logits, dim=-1)
        
        preds_db.extend(predictions.cpu().tolist())
        proba_db_test.extend(probs[:, 1].cpu().tolist())

proba_db_test = np.array(proba_db_test)

submission_db = pd.DataFrame({
    'id': test['id'],
    'label': preds_db
})

submission_db.to_csv('submission_distilbert.csv', index=False)
print('Submission guardada como submission_distilbert.csv')
print()
print('Distribución:')
print(submission_db['label'].value_counts())
print()
print(f'F1-Macro Val: 0.6301')

Submission guardada como submission_distilbert.csv

Distribución:
label
1    2543
0    1293
Name: count, dtype: int64

F1-Macro Val: 0.6301


In [24]:
# Obtener probabilidades de DistilBERT en validación
model_db.load_state_dict(torch.load('best_model_db.pt'))
model_db.eval()

proba_db_val = []
with torch.no_grad():
    for batch in val_loader_db:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model_db(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        proba_db_val.extend(probs[:, 1].cpu().tolist())

proba_db_val = np.array(proba_db_val)

# Probar ensemble de los tres
print('Ensemble RoBERTa + DistilBERT + NB:')
print()
best_f1_triple = 0
best_config = None

for w_rob in [0.5, 0.6, 0.7, 0.8]:
    for w_db in [0.1, 0.2, 0.3]:
        w_nb = round(1 - w_rob - w_db, 2)
        if w_nb < 0:
            continue
        
        proba_ens_val = w_rob * proba_roberta_val + w_db * proba_db_val + w_nb * nb_model.predict_proba(X_train_nb[-len(labels_val_all):])[: ,1]
        
        for threshold in [0.45, 0.50, 0.55]:
            y_pred = (proba_ens_val >= threshold).astype(int)
            f1 = f1_score(labels_val_all, y_pred, average='macro')
            if f1 > best_f1_triple:
                best_f1_triple = f1
                best_config = (w_rob, w_db, w_nb, threshold)

print(f'Mejor configuración:')
print(f'  RoBERTa={best_config[0]}, DistilBERT={best_config[1]}, NB={best_config[2]}')
print(f'  Threshold={best_config[3]}')
print(f'  F1-Macro Val={best_f1_triple:.4f}')
print(f'  Ensemble RoBERTa+NB anterior: 0.6463')

Ensemble RoBERTa + DistilBERT + NB:

Mejor configuración:
  RoBERTa=0.8, DistilBERT=0.1, NB=0.1
  Threshold=0.55
  F1-Macro Val=0.6454
  Ensemble RoBERTa+NB anterior: 0.6463


In [25]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Tokenizador RoBERTa-large
tokenizer_large = RobertaTokenizer.from_pretrained('roberta-large')

# Recrear datasets
train_dataset_large = FakeNewsDataset2(train_textos_tr, y_tr, tokenizer_large)
val_dataset_large = FakeNewsDataset2(train_textos_val, y_val, tokenizer_large)
test_dataset_large = FakeNewsDataset2(test_textos, tokenizer=tokenizer_large)

train_loader_large = DataLoader(train_dataset_large, batch_size=8, shuffle=True)  # batch más pequeño por memoria
val_loader_large = DataLoader(val_dataset_large, batch_size=16, shuffle=False)
test_loader_large = DataLoader(test_dataset_large, batch_size=16, shuffle=False)

print(f'Train: {len(train_dataset_large)} ejemplos, {len(train_loader_large)} batches')
print(f'Val:   {len(val_dataset_large)} ejemplos')
print(f'Test:  {len(test_dataset_large)} ejemplos')

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

Train: 7604 ejemplos, 951 batches
Val:   1343 ejemplos
Test:  3836 ejemplos


In [26]:
model_large = RobertaForSequenceClassification.from_pretrained(
    'roberta-large',
    num_labels=2
)
model_large = model_large.to(device)

optimizer_large = AdamW(model_large.parameters(), lr=1e-5, weight_decay=0.01)

epochs_large = 4
total_steps_large = len(train_loader_large) * epochs_large
scheduler_large = get_linear_schedule_with_warmup(
    optimizer_large,
    num_warmup_steps=int(0.1 * total_steps_large),
    num_training_steps=total_steps_large
)

print(f'Parámetros totales: {sum(p.numel() for p in model_large.parameters()):,}')

best_f1_large = 0

for epoch in range(epochs_large):
    model_large.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(train_loader_large):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer_large.zero_grad()
        outputs = model_large(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model_large.parameters(), 1.0)
        optimizer_large.step()
        scheduler_large.step()
        total_loss += loss.item()
        
        if batch_idx % 150 == 0:
            print(f'Epoch {epoch+1}/{epochs_large} | Batch {batch_idx}/{len(train_loader_large)} | Loss: {loss.item():.4f}')
    
    avg_loss = total_loss / len(train_loader_large)
    val_f1, _, _ = evaluar(model_large, val_loader_large)
    
    print(f'\nEpoch {epoch+1} completado | Avg Loss: {avg_loss:.4f} | Val F1-Macro: {val_f1:.4f}')
    
    if val_f1 > best_f1_large:
        best_f1_large = val_f1
        torch.save(model_large.state_dict(), 'best_model_large.pt')
        print(f'Mejor modelo guardado! F1: {best_f1_large:.4f}')
    print()

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parámetros totales: 355,361,794
Epoch 1/4 | Batch 0/951 | Loss: 0.8578
Epoch 1/4 | Batch 150/951 | Loss: 0.4893
Epoch 1/4 | Batch 300/951 | Loss: 0.7132
Epoch 1/4 | Batch 450/951 | Loss: 0.7231
Epoch 1/4 | Batch 600/951 | Loss: 0.8821
Epoch 1/4 | Batch 750/951 | Loss: 0.4111
Epoch 1/4 | Batch 900/951 | Loss: 0.8771

Epoch 1 completado | Avg Loss: 0.6518 | Val F1-Macro: 0.6129
Mejor modelo guardado! F1: 0.6129

Epoch 2/4 | Batch 0/951 | Loss: 0.6394
Epoch 2/4 | Batch 150/951 | Loss: 0.2460
Epoch 2/4 | Batch 300/951 | Loss: 1.4851
Epoch 2/4 | Batch 450/951 | Loss: 0.7064
Epoch 2/4 | Batch 600/951 | Loss: 0.4424
Epoch 2/4 | Batch 750/951 | Loss: 0.3228
Epoch 2/4 | Batch 900/951 | Loss: 0.5073

Epoch 2 completado | Avg Loss: 0.5927 | Val F1-Macro: 0.6375
Mejor modelo guardado! F1: 0.6375

Epoch 3/4 | Batch 0/951 | Loss: 0.4404
Epoch 3/4 | Batch 150/951 | Loss: 0.5190
Epoch 3/4 | Batch 300/951 | Loss: 0.5459
Epoch 3/4 | Batch 450/951 | Loss: 0.4968
Epoch 3/4 | Batch 600/951 | Loss: 0.0914
E

In [27]:
# Cargar mejor modelo RoBERTa-large
model_large.load_state_dict(torch.load('best_model_large.pt'))
model_large.eval()

preds_large = []
proba_large_test = []
with torch.no_grad():
    for batch in test_loader_large:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model_large(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(outputs.logits, dim=-1)
        
        preds_large.extend(predictions.cpu().tolist())
        proba_large_test.extend(probs[:, 1].cpu().tolist())

proba_large_test = np.array(proba_large_test)

submission_large = pd.DataFrame({
    'id': test['id'],
    'label': preds_large
})

submission_large.to_csv('submission_roberta_large.csv', index=False)
print('Submission guardada.')
print()
print('Distribución:')
print(submission_large['label'].value_counts())
print()
print(f'F1-Macro Val: 0.6446')

Submission guardada.

Distribución:
label
1    2569
0    1267
Name: count, dtype: int64

F1-Macro Val: 0.6446


In [1]:
# Probabilidades RoBERTa-large en validación
proba_large_val = []
with torch.no_grad():
    for batch in val_loader_large:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model_large(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        proba_large_val.extend(probs[:, 1].cpu().tolist())

proba_large_val = np.array(proba_large_val)

# Probar ensemble RoBERTa-large + NB
print('Ensemble RoBERTa-large + NB:')
print()
for w_large in [0.80, 0.85, 0.90, 0.95]:
    w_nb = round(1 - w_large, 2)
    proba_ens_val = w_large * proba_large_val + w_nb * nb_model.predict_proba(X_train_nb[-len(labels_val_all):])[: ,1]
    
    for threshold in [0.45, 0.50, 0.55]:
        y_pred = (proba_ens_val >= threshold).astype(int)
        f1 = f1_score(labels_val_all, y_pred, average='macro')
        print(f'Large={w_large:.2f} NB={w_nb:.2f} threshold={threshold}: F1-Val={f1:.4f}')
    print()

NameError: name 'torch' is not defined